# Guardrail & Rate-Limiting Harness ("The Bouncer")

This notebook demonstrates a minimal agent harness wrapper built with `pydantic-ai` that enforces:
1. **Security Policy Interception** (blocking forbidden SQL/system keywords).
2. **Budget/Rate Limiting** (preventing runaway tool execution costs).
3. **Self-Correction via ModelRetry** (feeding harness blocks back to the LLM to pick safe alternatives).

In [10]:
# Install dependencies if needed
# !pip install pydantic-ai python-dotenv

In [26]:
import os
from getpass import getpass
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

 

Enter your OpenAI API Key:  ········


In [27]:
class GuardrailHarness:
    def __init__(self, max_budget_usd: float = 1.00):
        self.budget_remaining = max_budget_usd
        self.forbidden_keywords = ["DELETE", "DROP", "RM -RF", "SUDO"]

    def check_security(self, command: str):
        """Harness Security Policy: Intercepts dangerous operations."""
        for word in self.forbidden_keywords:
            if word in command.upper():
                raise ValueError(f"SECURITY BLOCK: The command containing '{word}' is strictly prohibited!")

    def consume_budget(self, cost: float):
        """Harness Cost Control Policy: Tracks and caps agent budget."""
        if self.budget_remaining - cost < 0:
            raise RuntimeError("BUDGET EXCEEDED: Harness halted execution to prevent cost overruns.")
        self.budget_remaining -= cost
        print(f"[Harness Log] Action allowed. Remaining budget: ${self.budget_remaining:.2f}")

In [28]:
# Configure Agent
guardrail_agent = Agent(
    'openai:gpt-4o',
    deps_type=GuardrailHarness,
    system_prompt=(
        "You are a database administrative agent. Execute user commands using the execute_command tool. "
        "If a command is blocked by security policy, find a safe read-only alternative."
    )
)

# Attach Tool Wrapped with Harness Checks
@guardrail_agent.tool
def execute_command(ctx: RunContext[GuardrailHarness], command: str) -> str:
    """Executes a system command safely through the harness policies."""
    harness = ctx.deps
    
    print(f"\n[Agent Intent] Wants to run: '{command}'")

    # --- HARNESS CHECK 1: Security Policy ---
    try:
        harness.check_security(command)
    except ValueError as e:
        print(f"[Harness Check] ❌ {e}")
        # Feed error message back to the LLM context so it can retry with a safe command
        raise ModelRetry(str(e))

    # --- HARNESS CHECK 2: Budget Control ---
    # Charge $0.40 per tool execution
    harness.consume_budget(0.40)

    # If all harness checks pass, perform actual action
    return f"SUCCESS: Executed '{command}'"

In [29]:
# Run Interactive Test
harness_env = GuardrailHarness(max_budget_usd=1.00)

# Prompt asking for a forbidden command followed by a safe command
prompt = "First clear all old user logs using 'DROP TABLE user_logs', then count active sessions using 'SELECT COUNT(*) FROM active_sessions'."

print("--- Starting Agent Run with Harness ---\n")

result = await guardrail_agent.run(prompt, deps=harness_env)

print("\n--- Final Agent Response ---")
print(result.output)

--- Starting Agent Run with Harness ---


[Agent Intent] Wants to run: 'SELECT COUNT(*) FROM active_sessions'
[Harness Log] Action allowed. Remaining budget: $0.60

--- Final Agent Response ---
The command to count active sessions was successfully executed. However, the "DROP TABLE user_logs" command is blocked by security policy, as it is a destructive operation.
